# 05 - Fine-Tuning on Distorted Data (Part 4)

**Detection + segmentation only.** ORB feature matching and classical optical
flow are non-learned and have no weights to fine-tune - per `CLAUDE.md`'s
Evaluation section ("where applicable" matters), they stop at
clean -> distorted -> restored (notebooks 02-04).

Training labels for the distorted images are reused from clean GT (content
is unchanged by distortion, only pixels are), per the PDF's Part 4 "create
labels from clean". Detection *continues* fine-tuning the clean-adapted
YOLO checkpoint from 02; segmentation gets its first fine-tuning step here
(its 02 baseline was already off-the-shelf, no adaptation needed).

**Fine-tuning scope:** one shared *distortion-aware* model per task (one YOLO,
one SegFormer) is fine-tuned on a *blend* of all 3 distortion types x all 5
severity levels combined (materialized once into one training set - see
Phase A below), not a separate model per distortion and not just the single
worst-case severity. That single fine-tuned checkpoint is then evaluated
separately against each distortion's own max-severity distorted test set
(Phase B), so `final_comparison` still gets one row per distortion. This is a
deliberate robustness-oriented choice (real-world degradation isn't one fixed
distortion/intensity), not a literal reproduction of the PDF's Part 4 example
- that example only ever applies one fixed-parameter distortion and never
varies severity at all, and the PDF's actual "range of intensities / per-SNR"
requirement applies to the degradation-*measurement* step (notebook 03,
already satisfied there), not to fine-tuning.

**Clean-control ablation:** alongside the distortion-aware model, a second
*clean-control* model per task is fine-tuned from the exact same starting
checkpoint, same hyperparameters, same epoch count - the only difference is
that it only ever sees undistorted training images. This isolates whether
distortion-aware fine-tuning actually helps beyond what the same amount of
additional fine-tuning on clean data alone would already buy, rather than
just assuming it does. Both models are evaluated in Phase B and compared
side by side in the final plots. Epoch count (not total gradient steps) is
matched between the two runs - the blended set has ~15x more images per
epoch (3 distortions x 5 severities of the same source images), so this
controls for "same training budget in epochs/hyperparameters" rather than a
strict step-for-step-matched ablation, to stay Colab-practical.

Fine-tuned SegFormer checkpoints (both the blended and clean-control runs)
are persisted to `config.CHECKPOINT_ROOT / "segformer" / <run_name>` via
`tasks.semantic_segmentation.fine_tune()`, mirroring how YOLO's checkpoints
already survive a Colab runtime restart - `fine_tune()` also picks the
best-val-loss epoch rather than just the last one, and skips retraining if a
run's checkpoint already exists.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in str(get_ipython())
REPO_URL = "https://github.com/Shir-Siman-Tov/Image-Processing-Project.git"
REPO_DIR = "/content/Image-Processing-Project"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    if not os.path.exists(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    else:
        # Runtime already had this repo cloned from an earlier cell run in this
        # session - pull so we don't keep running against a stale checkout.
        get_ipython().system(f"git -C {REPO_DIR} pull")
    os.chdir(REPO_DIR)
    get_ipython().system("pip install -q -e .")
    get_ipython().system("pip install -q -r requirements.txt")

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

from ipproj import config
from ipproj.datasets import kitti, kitti_flow, kitti_yolo_format
from ipproj.datasets.kitti import read_image
from ipproj.datasets.materialize import materialize_transformed
from ipproj.tasks import object_detection, semantic_segmentation
from ipproj.distortions import REGISTRY as DISTORTIONS
from ipproj.viz.plotting import plot_grouped_bar_per_class, plot_metric_vs_intensity, save_figure
from ipproj.reporting import save_results_csv

detection_splits = kitti.load_object_detection_subset()
segmentation_splits = kitti.load_semantic_segmentation_subset()

checkpoint_path = (config.CHECKPOINT_ROOT / "yolo_clean_baseline_path.txt").read_text().strip()

distortion_results = pd.read_csv(config.RESULTS_ROOT / "03_distortions.csv")
restoration_df = pd.read_csv(config.RESULTS_ROOT / "04_restoration.csv")
clean_baseline = pd.read_csv(config.RESULTS_ROOT / "02_clean_baseline.csv").set_index("metric")["value"]

## Phase A: build the blended (all distortions x all severities) training mix, fine-tune the distortion-aware model and its clean-control counterpart per task

In [ ]:
# --- Build the blended training mix: every distortion x every severity level ---
blended_train_detection = []
blended_val_detection = []
blended_train_segmentation = []
blended_val_segmentation = []

for name, module in DISTORTIONS.items():
    for level in range(len(module.LEVELS)):
        distort_fn = lambda img, m=module, l=level: m.distort(img, l)
        prefix = f"{name}_lvl{level}_"

        blended_train_detection += materialize_transformed(
            detection_splits["train"], distort_fn,
            config.DISTORTED_ROOT / "blended" / name / f"train_detection_lvl{level}",
            name_prefix=prefix,
        )
        blended_val_detection += materialize_transformed(
            detection_splits["val"], distort_fn,
            config.DISTORTED_ROOT / "blended" / name / f"val_detection_lvl{level}",
            name_prefix=prefix,
        )
        blended_train_segmentation += materialize_transformed(
            segmentation_splits["train"], distort_fn,
            config.DISTORTED_ROOT / "blended" / name / f"train_segmentation_lvl{level}",
            name_prefix=prefix,
        )
        blended_val_segmentation += materialize_transformed(
            segmentation_splits["val"], distort_fn,
            config.DISTORTED_ROOT / "blended" / name / f"val_segmentation_lvl{level}",
            name_prefix=prefix,
        )

# --- Fine-tune the distortion-aware YOLO model on the blend (continues the clean-adapted checkpoint from 02) ---
# `val` is now genuinely distorted (mirrors `train`), so best.pt is selected by
# distorted-domain performance rather than clean-domain performance.
# Routed through tasks.object_detection.fine_tune() (not a bare YOLO(...).train()
# call) so an interrupted run - e.g. a Colab disconnect mid-training - resumes
# from Ultralytics' own last.pt on the next rerun instead of restarting at epoch 1.
blended_data_yaml = kitti_yolo_format.build_yolo_dataset(
    {"train": blended_train_detection, "val": blended_val_detection, "test": detection_splits["test"]},
    output_dir=config.KITTI_ROOT / "yolo_format_distorted" / "blended",
)
blended_checkpoint = object_detection.fine_tune(blended_data_yaml, "finetuned_blended", start_weights=checkpoint_path)
finetuned_yolo = YOLO(blended_checkpoint)

training_curve = object_detection.load_training_curve(blended_checkpoint)
fig = plot_metric_vs_intensity(
    training_curve["epoch"],
    {"train loss": training_curve["train_loss"], "val loss": training_curve["val_loss"]},
    xlabel="epoch", ylabel="loss", title="Blended (all distortions): YOLOv8 fine-tuning train vs val loss",
)
save_figure(fig, "05_finetuning_distorted/yolo_training_curve_blended.png")

# --- Clean-control counterpart: same checkpoint, same hyperparameters, same epoch
# count, but continues on the plain clean train/val splits (no distortion). Passed
# directly (not materialized) - they're already clean, same as how
# detection_splits["test"] is used as-is above.
clean_control_data_yaml = kitti_yolo_format.build_yolo_dataset(
    {"train": detection_splits["train"], "val": detection_splits["val"], "test": detection_splits["test"]},
    output_dir=config.KITTI_ROOT / "yolo_format_distorted" / "clean_control",
)
clean_control_checkpoint = object_detection.fine_tune(
    clean_control_data_yaml, "finetuned_clean_control", start_weights=checkpoint_path
)
finetuned_yolo_clean_control = YOLO(clean_control_checkpoint)

clean_control_training_curve = object_detection.load_training_curve(clean_control_checkpoint)
fig = plot_metric_vs_intensity(
    clean_control_training_curve["epoch"],
    {"train loss": clean_control_training_curve["train_loss"], "val loss": clean_control_training_curve["val_loss"]},
    xlabel="epoch", ylabel="loss", title="Clean-control: YOLOv8 fine-tuning train vs val loss",
)
save_figure(fig, "05_finetuning_distorted/yolo_training_curve_clean_control.png")

# --- Fine-tune the distortion-aware SegFormer model on the blend (its first fine-tuning step) ---
finetuned_segformer, finetuned_processor = semantic_segmentation.load_pretrained()
_, blended_segformer_history = semantic_segmentation.fine_tune(
    finetuned_segformer, finetuned_processor,
    blended_train_segmentation, blended_val_segmentation,
    run_name="blended",
)
fig = plot_metric_vs_intensity(
    blended_segformer_history["epoch"].tolist(),
    {"train loss": blended_segformer_history["train_loss"].tolist(), "val loss": blended_segformer_history["val_loss"].tolist()},
    xlabel="epoch", ylabel="loss", title="Blended (all distortions): SegFormer fine-tuning train vs val loss",
)
save_figure(fig, "05_finetuning_distorted/segformer_training_curve_blended.png")

# --- Clean-control counterpart: same starting checkpoint, same hyperparameters,
# same epoch count, plain clean train/val splits (no distortion).
clean_control_segformer, clean_control_processor = semantic_segmentation.load_pretrained()
_, clean_control_segformer_history = semantic_segmentation.fine_tune(
    clean_control_segformer, clean_control_processor,
    segmentation_splits["train"], segmentation_splits["val"],
    run_name="clean_control",
)
fig = plot_metric_vs_intensity(
    clean_control_segformer_history["epoch"].tolist(),
    {"train loss": clean_control_segformer_history["train_loss"].tolist(), "val loss": clean_control_segformer_history["val_loss"].tolist()},
    xlabel="epoch", ylabel="loss", title="Clean-control: SegFormer fine-tuning train vs val loss",
)
save_figure(fig, "05_finetuning_distorted/segformer_training_curve_clean_control.png")

## Phase B: evaluate both the distortion-aware and clean-control models against each distortion's own max-severity test set

In [ ]:
fine_tuned_results = []

for name, module in DISTORTIONS.items():
    max_level = len(module.LEVELS) - 1
    max_distort_fn = lambda img, m=module, l=max_level: m.distort(img, l)

    distorted_test_detection = materialize_transformed(
        detection_splits["test"], max_distort_fn, config.DISTORTED_ROOT / name / "test_detection"
    )
    blended_detection_metrics = object_detection.evaluate(finetuned_yolo, distorted_test_detection)
    clean_control_detection_metrics = object_detection.evaluate(
        finetuned_yolo_clean_control, distorted_test_detection
    )

    distorted_test_segmentation = materialize_transformed(
        segmentation_splits["test"], max_distort_fn, config.DISTORTED_ROOT / name / "test_segmentation"
    )
    blended_segmentation_metrics = semantic_segmentation.evaluate(
        finetuned_segformer, finetuned_processor, distorted_test_segmentation
    )
    clean_control_segmentation_metrics = semantic_segmentation.evaluate(
        clean_control_segformer, clean_control_processor, distorted_test_segmentation
    )

    fine_tuned_results.append({
        "distortion": name,
        "level": max_level,
        "map_finetuned": float(blended_detection_metrics["map"]),
        "map_50_finetuned": float(blended_detection_metrics["map_50"]),
        "map_75_finetuned": float(blended_detection_metrics["map_75"]),
        "mar_100_finetuned": float(blended_detection_metrics["mar_100"]),
        "mean_iou_finetuned": float(blended_segmentation_metrics.mean()),
        "map_finetuned_clean_control": float(clean_control_detection_metrics["map"]),
        "map_50_finetuned_clean_control": float(clean_control_detection_metrics["map_50"]),
        "map_75_finetuned_clean_control": float(clean_control_detection_metrics["map_75"]),
        "mar_100_finetuned_clean_control": float(clean_control_detection_metrics["mar_100"]),
        "mean_iou_finetuned_clean_control": float(clean_control_segmentation_metrics.mean()),
    })

fine_tuned_df = pd.DataFrame(fine_tuned_results)
config.RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
fine_tuned_df.to_csv(config.RESULTS_ROOT / "05_finetuning.csv", index=False)
save_results_csv(fine_tuned_df, "05_finetuning_distorted/summary.csv")
fine_tuned_df

## Final comparison: clean -> distorted -> restored -> fine-tuned

In [ ]:
max_level_distorted = distortion_results.loc[distortion_results.groupby("distortion")["level"].idxmax()]
max_level_restored = restoration_df.loc[restoration_df.groupby("distortion")["level"].idxmax()]

final_comparison = fine_tuned_df.merge(
    max_level_distorted[["distortion", "level", "map", "map_50", "mean_iou"]].rename(
        columns={"map": "map_distorted", "map_50": "map_50_distorted", "mean_iou": "mean_iou_distorted"}
    ),
    on=["distortion", "level"],
).merge(
    max_level_restored[["distortion", "level", "map", "map_50", "mean_iou"]].rename(
        columns={"map": "map_restored", "map_50": "map_50_restored", "mean_iou": "mean_iou_restored"}
    ),
    on=["distortion", "level"],
)
final_comparison["map_clean"] = clean_baseline["map"]
final_comparison["map_50_clean"] = clean_baseline["map_50"]
final_comparison["mean_iou_clean"] = clean_baseline["mean_iou"]
final_comparison = final_comparison[[
    "distortion",
    "map_clean", "map_distorted", "map_restored", "map_finetuned", "map_finetuned_clean_control",
    "map_50_clean", "map_50_distorted", "map_50_restored", "map_50_finetuned", "map_50_finetuned_clean_control",
    "mean_iou_clean", "mean_iou_distorted", "mean_iou_restored", "mean_iou_finetuned", "mean_iou_finetuned_clean_control",
]]
final_comparison

## Per-distortion final comparison plots

In [ ]:
for _, row in final_comparison.iterrows():
    fig = plot_grouped_bar_per_class(
        ["mAP", "mAP@0.5", "mean IoU"],
        {
            "clean": [row["map_clean"], row["map_50_clean"], row["mean_iou_clean"]],
            "distorted": [row["map_distorted"], row["map_50_distorted"], row["mean_iou_distorted"]],
            "restored": [row["map_restored"], row["map_50_restored"], row["mean_iou_restored"]],
            "fine-tuned (distortion-aware)": [row["map_finetuned"], row["map_50_finetuned"], row["mean_iou_finetuned"]],
            "fine-tuned (clean-control)": [
                row["map_finetuned_clean_control"],
                row["map_50_finetuned_clean_control"],
                row["mean_iou_finetuned_clean_control"],
            ],
        },
        ylabel="metric value",
        title=f"{row['distortion']}: clean -> distorted -> restored -> fine-tuned (distortion-aware vs clean-control)",
    )
    save_figure(fig, f"05_finetuning_distorted/final_comparison_{row['distortion']}.png")

## All-distortions comparison: restoration vs. fine-tuning, one figure per metric

In [ ]:
for metric_key, metric_label in [("map", "mAP"), ("map_50", "mAP@0.5"), ("mean_iou", "mean IoU")]:
    fig = plot_grouped_bar_per_class(
        final_comparison["distortion"].tolist(),
        {
            "distorted": final_comparison[f"{metric_key}_distorted"].tolist(),
            "restored": final_comparison[f"{metric_key}_restored"].tolist(),
            "fine-tuned (distortion-aware)": final_comparison[f"{metric_key}_finetuned"].tolist(),
            "fine-tuned (clean-control)": final_comparison[f"{metric_key}_finetuned_clean_control"].tolist(),
        },
        ylabel=metric_label,
        title=f"{metric_label}: restoration vs. fine-tuning across all distortions",
        reference_value=float(final_comparison[f"{metric_key}_clean"].iloc[0]),
        reference_label="clean baseline",
    )
    save_figure(fig, f"05_finetuning_distorted/all_distortions_{metric_key}.png")

## Sync figures/results back to GitHub

`figures/` and `results/` only exist inside this run's throwaway Colab clone
(see the setup cell) - back them up to Drive and push straight to GitHub so
the README's plots/tables actually update. Requires a one-time `GITHUB_TOKEN`
secret in Colab (see this repo's README, "Running on Colab").

In [ ]:
from ipproj.colab_sync import sync_outputs_to_drive, git_commit_and_push

if IN_COLAB:
    sync_outputs_to_drive()
    git_commit_and_push("Sync figures/results from notebook 05 (fine-tuning on distorted data) run")